# Web Data Project - Knowledge Graph Pipeline
# Yuhan XUE - Anthony YANG - François ZAPLETAL
## British Royal Family (19th century)

- **Module 1** m1_data_acquisition.py - Web Crawling & NER
- **Module 2** m2_kb_construction.py - KB Construction, Alignment & Expansion
- **Module 3** m3_reasoning.py - SWRL Reasoning
- **Module 4** m4_kge.py - Knowledge Graph Embeddings
- **Module 5** m5_rag.py - RAG over RDF/SPARQL


In [1]:
# Install all dependencies
# !pip install trafilatura spacy requests sentence-transformers rdflib owlready2 pykeen torch scikit-learn matplotlib seaborn pandas
# !python -m spacy download en_core_web_md
# ollama run llama3:latest  (in the terminal after download ollama)


## MODULE 1 - Data Acquisition & Information Extraction

In [2]:
from src.m1_data_acquisition import *

# Seed URLs - British Royal Family (19th century)
seed_urls = [
    # Level 1: Encyclopaedic (rich syntax, nested clauses)
    "https://en.wikipedia.org/wiki/Queen_Victoria",
    "https://en.wikipedia.org/wiki/Prince_Albert_of_Saxe-Coburg_and_Gotha",
    "https://en.wikipedia.org/wiki/Victoria,_Princess_Royal",
    "https://en.wikipedia.org/wiki/Edward_VII",
    "https://en.wikipedia.org/wiki/Princess_Alice_of_the_United_Kingdom",
    "https://en.wikipedia.org/wiki/Alfred,_Duke_of_Saxe-Coburg_and_Gotha",
    "https://en.wikipedia.org/wiki/Princess_Helena_of_the_United_Kingdom",
    "https://en.wikipedia.org/wiki/Princess_Louise,_Duchess_of_Argyll",
    "https://en.wikipedia.org/wiki/Prince_Arthur,_Duke_of_Connaught_and_Strathearn",
    "https://en.wikipedia.org/wiki/Prince_Leopold,_Duke_of_Albany",
    "https://en.wikipedia.org/wiki/Princess_Beatrice_of_the_United_Kingdom",
    "https://en.wikipedia.org/wiki/Prince_Edward,_Duke_of_Kent_and_Strathearn",
    "https://en.wikipedia.org/wiki/Princess_Victoria_of_Saxe-Coburg-Saalfeld",
    # Level 2: Official / narrative sources
    "https://www.royal.uk/queen-victoria",
    "https://www.royal.uk/edward-vii",
    "https://www.biography.com/royalty/queen-victoria",
    "https://www.biography.com/royalty/king-edward-vii",
    "https://www.historic-uk.com/HistoryUK/KingsQueensofBritain/Queen-Victoria/",
    # Level 3: Educational (simple direct sentences - guarantees core triplets)
    "https://www.ducksters.com/biography/women_leaders/queen_victoria.php",
    "https://kids.britannica.com/kids/article/Queen-Victoria/353896",
    "https://kids.britannica.com/students/article/Queen-Victoria/277598",
    "https://www.bbc.co.uk/bitesize/articles/zfd8jhv",
    "https://www.natgeokids.com/uk/discover/history/monarchy/queen-victoria/"
]

output_file_jsonl = "../data/raw_and_processed/crawler_output.jsonl"
init_file = True

for url in seed_urls:
    scraping_site(url, output_file_jsonl, nb_leafs=10, nb_cara=100, new=init_file)
    init_file = False
    time.sleep(random.uniform(1.5, 3.0))

Output saved in ../data/raw_and_processed/crawler_output.jsonl


In [3]:
from src.m1_data_acquisition import name_entity_recognition

# NER quick test
txt_2 = "Paris is located in France and Emmanuel Macron knows it"
txt_3 = "Father: The Kangxi Emperor (of whom he was the 4th son)"
txt_4 = "Queen Elizabeth II is the mother of Charles"
print(name_entity_recognition(txt_2))
print(name_entity_recognition(txt_3))
print(name_entity_recognition(txt_4))

[{'subject': 'Paris', 'subject_type': 'GPE', 'relation': 'locate in', 'object': 'France', 'object_type': 'GPE', 'context': 'Paris is located in France and Emmanuel Macron knows it'}]
[]
[{'subject': 'Elizabeth II', 'subject_type': 'PERSON', 'relation': 'be mother of', 'object': 'Charles', 'object_type': 'PERSON', 'context': 'Queen Elizabeth II is the mother of Charles'}]


In [4]:
from src.m1_data_acquisition import extract_knowledge

# Extract triplets from crawler output and save to CSV
output_file_csv = "../data/raw_and_processed/extracted_knowledge.csv"
extract_knowledge(output_file_jsonl, output_file_csv)


Output saved: ../data/raw_and_processed/extracted_knowledge.csv
Total validated relations: 369
Total rejected relations: 0



## MODULE 2 - KB Construction, Alignment & Expansion

In [5]:
from src.m2_kb_construction import build_initial_kb

# Build the initial RDF knowledge base from extracted triplets
build_initial_kb("../data/raw_and_processed/extracted_knowledge.csv", "../kg_artifacts/initial_kb.ttl")

Knowledge base generated: ../kg_artifacts/initial_kb.ttl
Unique triplets: 360


In [6]:
from src.m2_kb_construction import (
    contextual_spotlight_linking,
    triple_based_predicate_alignment_approach_b,
    generate_global_alignment
)

# Entity linking with DBpedia Spotlight
contextual_spotlight_linking(
    "../data/raw_and_processed/extracted_knowledge.csv",
    "../data/raw_and_processed/mapping_entity.csv",
    "../kg_artifacts/entity_alignment.ttl"
)

# Predicate alignment (semantic soft scoring against DBpedia schema)
triple_based_predicate_alignment_approach_b(
    "../data/raw_and_processed/extracted_knowledge.csv",
    "../data/raw_and_processed/mapping_entity.csv",
    "../data/raw_and_processed/output_predicate.csv"
)

# Merge entity and predicate alignments into one global TTL file
generate_global_alignment(
    "../data/raw_and_processed/mapping_entity.csv",
    "../data/raw_and_processed/output_predicate.csv",
    "../kg_artifacts/alignment.ttl"
)

[attend] -> dbo:numberOfPeopleAttending (number of people attending) [Score: 0.60]
[be former] -> dbo:formerName (former name) [Score: 0.56]

Template saved: ../data/raw_and_processed/output_predicate.csv
Building global alignment graph (Entities + Predicates)
Global alignment complete
- 252 entities aligned (owl:sameAs).
- 90 predicates aligned (owl:equivalentProperty).
- Output: ../kg_artifacts/alignment.ttl


In [7]:
from src.m2_kb_construction import generate_dynamic_ontology

# Generate the private OWL ontology
generate_dynamic_ontology(
    "../data/raw_and_processed/extracted_knowledge.csv",
    "../data/raw_and_processed/output_predicate.csv",
    "../kg_artifacts/ontology.ttl"
)

Private ontology generated: ../kg_artifacts/ontology.ttl


In [1]:
from src.m2_kb_construction import mass_semantic_expansion

# Expand the KB via DBpedia 2-hop SPARQL queries 
mass_semantic_expansion(
        "../kg_artifacts/initial_kb.ttl", "../data/raw_and_processed/mapping_entity.csv",
        "../kg_artifacts/expanded_kb.nt",
        densification_sample_ratio=0.02,
        confidence_threshold=0.6,
        similarity_threshold=0.25
)

C:\Users\zaple\anaconda3\envs\workspace\Lib\site-packages\rdflib\plugins\serializers\nt.py:39: UserWarning: NTSerializer always uses UTF-8 encoding. Given encoding was: None
  warnings.warn(


In [2]:
from src.m2_kb_construction import update_schema_and_sanitize_kb

# Clean, align and privatise the expanded KB
update_schema_and_sanitize_kb(
    "../kg_artifacts/expanded_kb.nt",
    "../kg_artifacts/ontology.ttl",
    "../kg_artifacts/alignment.ttl",
    "../kg_artifacts/cleaned_expanded_kb.nt",
    "../kg_artifacts/ontology_expanded.ttl",
    "../kg_artifacts/alignment_expanded.ttl"
)


Pipeline complete!
-> Final clean graph: 6137 triples.
-> Updated: ../kg_artifacts/ontology_expanded.ttl, ../kg_artifacts/alignment_expanded.ttl
-> New fact graph: ../kg_artifacts/cleaned_expanded_kb.nt



## MODULE 3 - Knowledge Reasoning with SWRL

In [3]:
from src.m3_reasoning import run_family_swrl

# Part 1: SWRL rule on family.owl - infer 'oldPerson' (age > 60)
# Expected: Peter (70) and Marie (69) are inferred as oldPerson
old_persons = run_family_swrl("../kg_artifacts/family.owl")
print(f"\n-> {len(old_persons)} individual(s) inferred as oldPerson")


Persons in ontology (12):
  - Tom        | age = 10
  - Thomas     | age = 40
  - Alex       | age = 25
  - Michael    | age = 5
  - Peter      | age = 70
  - Marie      | age = 69
  - Sylvie     | age = 30
  - John       | age = 45
  - Pedro      | age = 10
  - Claude     | age = 5
  - Chloé      | age = 18
  - Paul       | age = 38

SWRL rule registered in ontology:
  Person(?p) ∧ age(?p, ?a) ∧ swrlb:greaterThan(?a, 60)  oldPerson(?p)
  [Note: swrlb built-in applied via Python filter - OWLReady2 limitation]

RESULT - Individuals inferred as 'oldPerson' (2):
 Peter      | age = 70   classified as oldPerson
 Marie      | age = 69   classified as oldPerson
Persons not classified as oldPerson (age ≤ 60 or age unknown):
 Tom        | age = 10
 Thomas     | age = 40
 Alex       | age = 25
 Michael    | age = 5
 Sylvie     | age = 30
 John       | age = 45
 Pedro      | age = 10
 Claude     | age = 5
 Chloé      | age = 18
 Paul       | age = 38

-> 2 individual(s) inferred as oldPerson


In [4]:
from src.m3_reasoning import prepare_reasoning_base, run_royal_swrl_reasoning

# Part 2: SWRL rules on the Royal KB
# Merge ontology + cleaned KB into a single OWL file
prepare_reasoning_base(
    "../kg_artifacts/ontology_expanded.ttl",
    "../kg_artifacts/cleaned_expanded_kb.nt",
    "../kg_artifacts/reasoning_base.owl"
)

# Step B - Apply SWRL rules and run Pellet inference
# Rules: beMotherOf->parent, parent+parent->ancestor, spouses symmetry,
#        marry->spouses, successor->predecessor inverse
run_royal_swrl_reasoning("../kg_artifacts/reasoning_base.owl", "../kg_artifacts/reasoned_knowledge_base.owl")

* Owlready * Reparenting family.Chloé: {family.Female} => {family.Daughter, family.oldPerson}
* Owlready * Reparenting family.Sylvie: {family.Female} => {family.Daughter, family.oldPerson}
* Owlready * Reparenting family.Claude: {family.Female} => {family.Daughter, family.oldPerson}
* Owlready * Reparenting family.Pedro: {family.Male} => {family.Son, family.oldPerson}
* Owlready * Reparenting family.Thomas: {family.Male} => {family.Son, family.oldPerson, family.Father}
* Owlready * Reparenting family.Paul: {family.Male} => {family.Son, family.oldPerson}
* Owlready * Reparenting family.Michael: {family.Male} => {family.Son, family.oldPerson}
* Owlready * Reparenting family.Tom: {family.Male} => {family.Son, family.oldPerson}
* Owlready * Reparenting family.Alex: {family.Female} => {family.Female, family.oldPerson}
* Owlready * Reparenting family.Marie: {family.oldPerson, family.Female} => {family.Mother, family.oldPerson}
* Owlready * Reparenting family.Peter: {family.Male, family.oldPe

In [5]:
from src.m2_kb_construction import analyze_graph_health, export_for_kge, convert_to_turtle

# KB statistics dashboard
analyze_graph_health("../kg_artifacts/reasoned_knowledge_base.owl")

# Export train/valid/test for KGE
export_for_kge("../kg_artifacts/reasoned_knowledge_base.owl", "../data/kge_datasets/")

# Convert to Turtle for RAG pipeline
convert_to_turtle("../kg_artifacts/reasoned_knowledge_base.owl",
                  "../kg_artifacts/reasoned_knowledge_base.ttl")

Conversion successful: ../kg_artifacts/reasoned_knowledge_base.ttl (13829 triples)



## MODULE 4 - Knowledge Graph Embeddings

In [6]:
from src.m4_kge import run_full_kge_pipeline

# Full KGE pipeline:
#   - TransE vs DistMult training (identical config)
#   - MRR, Hits@1/3/10 evaluation
#   - Size sensitivity (20k / 50k / full)
#   - t-SNE coloured by ontology class
#   - Nearest neighbors
#   - Relation behavior analysis
#   - SWRL vs embedding comparison (Exercise 8)
kge_results = run_full_kge_pipeline(
    kge_data_folder="../data/kge_datasets/",
    embedding_dim=25,
    epochs=50,
    lr=0.01,
    batch_size=512,
    query_entities=[
        "VictoriaAlexandria", "Albert_II_of_Germany", "George_IV",
        "Albert_II_of_Germany", "Frederick_William_II_of_Prussia", "Victoria,_Princess_Royal"
    ]
)


  Top 10 most similar relation pairs (potential inverses/synonyms):
    predecessor                         <-> successor                           | sim = 0.9942
    marryOf                             <-> spouse                              | sim = 0.9941
    isChildOf                           <-> isParentOf                          | sim = 0.9848
    isolateFromOtherChildUnderSoCall    <-> raiseLargelyIsolateFromOtherChildUnderSoCall | sim = 0.9762
    relatives                           <-> restoreRelationWith                 | sim = 0.9647
    leaveForGreatSafetyOfHouse89PrivateEstateOn <-> spendAtOn                           | sim = 0.9620
    mother                              <-> parent                              | sim = 0.9581
    lowerHouse                          <-> occupyHouseAsHer                    | sim = 0.9550
    occupyHouseAsHer                    <-> upperHouse                          | sim = 0.9489
    beNowNeighbourBothAt                <-> stayAt         

In [7]:
from src.m4_kge import print_comparison_table

# Final metrics table for the report
print_comparison_table([kge_results["transe"], kge_results["distmult"]])

for model_key in ["transe", "distmult"]:
    r = kge_results[model_key]
    print(f"\n{r['model_name']}:")
    for metric, value in r["metrics"].items():
        print(f"  {metric} = {value:.4f}")

COMPARISON TABLE - TransE vs DistMult
Model        |      MRR |   Hits@1 |   Hits@3 |  Hits@10
TransE       |   0.1099 |   0.0299 |   0.1327 |   0.2670
DistMult     |   0.2204 |   0.1545 |   0.2553 |   0.3447


TransE:
  MRR = 0.1099
  Hits@1 = 0.0299
  Hits@3 = 0.1327
  Hits@10 = 0.2670

DistMult:
  MRR = 0.2204
  Hits@1 = 0.1545
  Hits@3 = 0.2553
  Hits@10 = 0.3447



## MODULE 5 - RAG over RDF/SPARQL

In [4]:
from src.m5_rag import run_benchmark

# Make sure Ollama is running: ollama serve
# and the model is pulled: ollama pull llama3:latest

TTL_FILE = "../kg_artifacts/reasoned_knowledge_base.ttl"

questions = [
    "Where is the birthplace of Lewis Carroll?",
    "Who is the birthbplace of Joseph Lebeau?",
    "Who is the succesor of George III?",
    "Who is the father of George IV?",
    "Who is the father of Prince Gustaf Adolf ?",
]

run_benchmark(TTL_FILE, questions, model="llama3:latest")

SPARQL generated:
SELECT ?answer WHERE {
  { ?entity priv:father ?answer . }
  UNION
  { ?answer priv:father ?entity . }
  FILTER(regex(str(?entity), "gustaf.*adolf", "i"))
}

Self-repair triggered? Yes
RAG ANSWER:
According to the provided context, Prince Gustaf Adolf's father is Carl XVI Gustaf.


In [2]:
from src.m5_rag import load_graph, get_schema_summary, graph_rag_pipeline

# Interactive single-question mode
g = load_graph("../kg_artifacts/reasoned_knowledge_base.ttl")
schema = get_schema_summary(g)

question = "Who is the succesor of George III?"
response, query, repaired = graph_rag_pipeline(question, g, schema, model="llama3:latest")

print(f"SPARQL:\n{query}\n")
print(f"Self-repair: {repaired}")
print(f"Answer: {response}")

SPARQL:
PREFIX dbpedia: <http://dbpedia.org/resource/>
PREFIX dbo: <http://dbpedia.org/ontology/>

ASK {
  ?person dbo:succeeded dbpedia:George_III .
}

Self-repair: True
Answer: George IV was the successor of George III.
